# Can we do kingdoms

In [ ]:
#| default_exp game/property

## The politcs of Geography
This is part of a larger game design project where I create a board game generator for my son. I created code that can load or maps with different elevations. from there we can run weather and climate simulators (as well as some terraforming to make sure that rivers wind up going somewhere). The next step is to try to create countries.

At some level, the mistakes are more interesting than the results. Because I use water as what matters, you can see why Sacramento became the capital of California. By no means does this have any historical or real accuracy. but I do think life is better with cool pictures

In [ ]:
#| export
import sys
import math
import numpy as np
import pandas as pd
import random
from fastcore.basics import patch
import heapq # for shortest path
from importlib import resources

In [ ]:
from dataclasses import dataclass, field , asdict
from typing import Optional, List, Set , Callable
from enum import Enum 
import uuid

In [ ]:
#| export
#This is kitchen sink approach to the library. I just didn't know what I needed

from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover


In [ ]:
#| export
from HexMagic.game.settlement import CountryFlag

In [ ]:
#| export
from HexMagic.weather import TerraDemo
from HexMagic.geology import  SoilSystem, DrainageBasins, Geology

In [ ]:
class Resources(Enum):
    """Available Foods for pieces"""
    GRAIN = "GRAIN"
    CHICKEN = "CHICKEN"
    VEGATABLES = "VEGATABLES"
    FRUIT = "FRUIT"
    FISH = "FISH"
    HERD = "HERD"
    

In [ ]:
@dataclass 
class Piece: 
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0  # Kingdom/country ID
    parent_id: Optional[str] = None  # ID of piece that spawned this
    location:int = None

    # Core attributes
    size: int = 100  # Number of units in this piece
    health: int = 100  # Hit points (0-100 scale)
    max_health: int = 100

    # Vision and intelligence
    sight: int = 3  # How many hex rings they can see
    memory: float = 0.8  # Retention rate (0-1, higher = better memory)

    # Movement
    movement_range: int = 4  # Max weighted hexes per turn
    current_position: int = -1  # Current hex index


    # Settlement state
    harvest_goal: Resources = None
    settle_progress: int = 0  # Turns spent settling (settlement complete at threshold)
    settle_threshold: int = 3
    db: GeoStorage = None # this should not be encoded or decoded
    flag:CountryFlag = None # this also does not need to be encoded

    PIECE_FIELDS = ['id','owner_id','parent_id','location','size','health',
                'max_health','sight','memory','movement_range',
                'current_position','harvest_goal','settle_progress','settle_threshold']

    def encode(self) -> str:
        """Encode piece as a single tab-delimited line."""
        vals = []
        for f in self.PIECE_FIELDS:
            v = getattr(self, f)
            if v is None:        vals.append('')
            elif isinstance(v, Resources): vals.append(v.value)
            else:                vals.append(str(v))
        return '\t'.join(vals)

    @staticmethod
    def decode(line: str) -> 'Piece':
        """Decode a tab-delimited line into a Piece."""
        parts = line.split('\t')
        kw = dict(zip(PIECE_FIELDS, parts))
        # Convert types back
        for f in ['owner_id','size','health','max_health','sight',
                'movement_range','current_position','settle_progress','settle_threshold']:
            kw[f] = int(kw[f]) if kw[f] else 0
        for f in ['location']:
            kw[f] = int(kw[f]) if kw[f] else None
        kw['memory'] = float(kw['memory']) if kw['memory'] else 0.8
        kw['parent_id'] = kw['parent_id'] or None
        kw['harvest_goal'] = Resources(kw['harvest_goal']) if kw['harvest_goal'] else None
        return Piece(**kw)



## Settlement

In [ ]:
@dataclass 
class Settlement: 
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0
    location: int = None
    name:str = ""
    # Core attributes
    size: int = 100
    health: int = 100
    citizens: List[Piece] = field(default_factory=list)
    db: GeoStorage = None # this should not be encoded or decoded
    flag:CountryFlag = None #this should not be encoded or decode

    SETTLEMENT_FIELDS = ['id', 'owner_id', 'name', 'location', 'size', 'health']

    def encode(self) -> str:
        """Encode settlement as tab-delimited lines. First line is settlement fields, remaining lines are citizen pieces."""
        vals = []
        for f in self.SETTLEMENT_FIELDS:
            v = getattr(self, f)
            vals.append('' if v is None else str(v))
        lines = ['\t'.join(vals)]
        for c in self.citizens:
            lines.append(c.encode())
        return '\n'.join(lines)

    @staticmethod
    def decode(text: str) -> 'Settlement':
        """Decode a settlement from encoded text. First line is settlement, remaining lines are citizen pieces."""
        lines = text.strip().split('\n')
        parts = lines[0].split('\t')
        kw = dict(zip(Settlement.SETTLEMENT_FIELDS, parts))
        for f in ['owner_id', 'size', 'health']:
            kw[f] = int(kw[f]) if kw[f] else 0
        kw['location'] = int(kw['location']) if kw['location'] else None
        citizens = [Piece.decode(l) for l in lines[1:] if l.strip()]
        return Settlement(**kw, citizens=citizens)


In [ ]:
@patch
def updateFlag(self:Settlement,flag):
    self.flag = flag
    for citizen in self.citizens:
        citizen.flag = flag

I need an encode and decode for Settlement. Can this be a dataclass

In [ ]:
#| export
from typing import NamedTuple

class WatershedFlow(NamedTuple):
    """ a place holder so we don't recompute flows which can be expensive """
    watershed: Watershed
    flow: float


   

In [ ]:
#| export
class TradeRoute:
    """ going to figure how to link up what we build"""

    def __init__(self, path: [HexPosition],  origin = 0, cost: float=0,  name: str=""):
        self.path = path
        self.cost = cost
        self.name = name
        self.origin = origin

    def destination(self)->HexPosition:
        return self.path[-1]

    @staticmethod
    def decode(s: str) -> 'TradeRoute':
        """Decode TradeRoute from string format."""
        name = ""
        origin = 0
        cost = 0.0
        path = []
        
        for line in s.strip().split('\n'):
            if ':' not in line:
                continue
            key, val = line.split(':', 1)
            if key == 'name':
                name = val
            elif key == 'origin':
                origin = int(val)
            elif key == 'cost':
                cost = float(val)
            elif key == 'path':
                path = [int(h) for h in val.split(',') if h]
        
        return TradeRoute(path=path, origin=origin, cost=cost, name=name)


In [ ]:
#| export
@patch
def encode(self: TradeRoute) -> str:
    """Encode TradeRoute to string format."""
    path_str = ','.join(str(h) for h in self.path)
    lines = [
        f"name:{self.name}",
        f"origin:{self.origin}",
        f"cost:{self.cost}",
        f"path:{path_str}"
    ]
    return '\n'.join(lines)

In [ ]:
#| export
@patch
def encode(self: HexRegion) -> str:
    """Encode HexRegion as comma-separated hex indices."""
    return ','.join(str(h) for h in sorted(self.hexes))

@staticmethod
def _decode_hexregion(s: str, hexGrid: HexGrid) -> HexRegion:
    """Decode HexRegion from comma-separated hex indices."""
    hexes = set(int(h) for h in s.split(',') if h)
    return HexRegion(hexes=hexes, hexGrid=hexGrid)

HexRegion.decode = _decode_hexregion


## Kingdom

In [ ]:
#| export
class Kingdom:
    """ This is a terrority run by a single government"""
    def __init__(self, capital_hex: int, region: HexRegion, world: Geology,countryId:int = 0, flag = None):
        capital = Settlement(owner_id=countryId,location=capital_hex)
        self.settlements = [capital]  # capital is first
        self.region = region
        self.world = world
        self.countryId = countryId
        self.captured:[WatershedFlow] = []
        self.routes:[TradeRoute] = []
        self.flag = flag
        self.countryName = ""
        self.db = None # this should not be encoded or decoded

        
    
    @classmethod
    def start(cls, world: Geology, scored:[WatershedFlow],palette_name: str = "husl"):
        terrain = world.terrain
        """Create initial kingdoms from best watersheds."""
        kingdoms = []
        i = 1 # lets reserve 0 for unclaimed and -1 for unavailable

    
        # Generate styles using seaborn palette
        flags = CountryFlag.seaborn(palette_name, levels=len(scored)+2) # just so we can index by 1
        saturation = 0.7

        for score in scored:
            ws = score.watershed
            capital = ws.max_flow_hex()[0]
          
            kingdom = cls(capital, ws.region, world, countryId=i,flag=flags[i])
            i += 1
            kingdom.captured.append(score)
            kingdoms.append(kingdom)
        
        return kingdoms

    @staticmethod
    def decode(s: str, world: Geology) -> 'Kingdom':
        """Decode Kingdom from string format."""
        lines = s.strip().split('\n')
        
        countryId = 0
        countryName = ""
        settlements = []
        flag = None
        region = None
        captured = []
        routes = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('countryId:'):
                countryId = int(line.split(':', 1)[1])
            elif line.startswith('countryName:'):
                countryName = line.split(':', 1)[1]
            elif line.startswith('flag:'):
                flag = CountryFlag.decode(line.split(':', 1)[1])
            elif line.startswith('+settlements'):
                i += 1
                while i < len(lines) and not lines[i].startswith('-settlements'):
                    if lines[i].startswith('+settlement'):
                        settle_lines = []
                        i += 1
                        while i < len(lines) and not lines[i].startswith('-settlement'):
                            settle_lines.append(lines[i])
                            i += 1
                        settlements.append(Settlement.decode('\n'.join(settle_lines)))
                    i += 1
            elif line.startswith('+region'):
                region_lines = []
                i += 1
                while i < len(lines) and not lines[i].startswith('-region'):
                    region_lines.append(lines[i])
                    i += 1
                region = HexRegion.decode('\n'.join(region_lines), world.terrain.hexGrid)
            elif line.startswith('+captured:'):
                i += 1
                while i < len(lines) and not lines[i].startswith('-captured'):
                    if lines[i].startswith('+watershed:'):
                        flow = float(lines[i].split(':', 1)[1])
                        ws_lines = []
                        i += 1
                        while i < len(lines) and not lines[i].startswith('-watershed'):
                            ws_lines.append(lines[i])
                            i += 1
                        ws = Watershed.decode('\n'.join(ws_lines), world.terrain)
                        captured.append(WatershedFlow(ws, flow))
                    i += 1
            elif line.startswith('+routes:'):
                i += 1
                route_lines = []
                while i < len(lines) and not lines[i].startswith('-routes'):
                    if lines[i] == '---':
                        if route_lines:
                            routes.append(TradeRoute.decode('\n'.join(route_lines)))
                        route_lines = []
                    else:
                        route_lines.append(lines[i])
                    i += 1
            i += 1
        
        # Build kingdom using first settlement's location as capital_hex
        capital_hex = settlements[0].location if settlements else 0
        kingdom = Kingdom(capital_hex, region, world, countryId, flag)
        kingdom.settlements = settlements
        kingdom.countryName = countryName
        kingdom.captured = captured
        kingdom.routes = routes
        
        return kingdom





In [ ]:
#| export
@patch
def encode(self: Kingdom) -> str:
    """Encode Kingdom to string format."""
    lines = [
        f"countryId:{self.countryId}",
        f"countryName:{self.countryName}",
    ]
    
    # Flag
    if self.flag:
        lines.append(f"flag:{self.flag.encode()}")
    
    # Settlements
    if self.settlements:
        lines.append(f"+settlements:{len(self.settlements)}")
        for s in self.settlements:
            lines.append("+settlement")
            lines.append(s.encode())
            lines.append("-settlement")
        lines.append("-settlements")
    
    # Region
    lines.append("+region")
    lines.append(self.region.encode())
    lines.append("-region")
    
    # Captured watersheds
    if self.captured:
        lines.append(f"+captured:{len(self.captured)}")
        for wf in self.captured:
            lines.append(f"+watershed:{wf.flow}")
            lines.append(wf.watershed.encode())
            lines.append("-watershed")
        lines.append("-captured")
    
    # Routes
    if self.routes:
        lines.append(f"+routes:{len(self.routes)}")
        for route in self.routes:
            lines.append(route.encode())
            lines.append("---")
        lines.append("-routes")
    
    return '\n'.join(lines)


## Details

class CountryDetails:
    def __init__(self,country:Kingdom):
        self.country = country
        self.updateParent(country.world.terrain)

    def updateParent(self,parent:Terrain):
        self.parent = parent
        grid, subregion, regionMapper = self.country.region.crop_to_centered_grid(style=StyleCSS("base",fill="lightgray",stroke="blue"))
        self.subregion = subregion
        self.regionMapper = regionMapper
        self.countryMap = Terrain(bounds = grid.bounds,
            radius = grid.radius,
            colorLevels= parent.colorLevels,
            seaLevel = parent.seaLevel,
            elevationDelta = parent.elevationDelta,
            geo = parent.geo,
            climate = parent.climate
        )
        numHexes = len(grid.hexes)
        self.countryMap.hexGrid = grid
        self.countryMap.elevations = np.zeros(numHexes)

        field_names = list(parent.fields.keys())
        
        for aField in field_names:
            self.countryMap.fields[aField] = np.zeros(numHexes)

        mapper = {}
        for dest in range(numHexes):
            source = regionMapper(dest)
            self.countryMap.elevations[dest] = parent.elevations[source]
            mapper[source] = dest
            for aField in field_names:
                self.countryMap.fields[aField][dest] = parent.fields[aField][source]
        self.mapper = mapper
       



@patch
def viewableRegion(self:CountryDetails,region)->HexRegion:
    """ return a region that have hexes that work """
    hexes = set([self.mapper[i] for i in region.hexes if i in self.mapper])
    return HexRegion(hexes=hexes, hexGrid = self.countryMap.hexGrid)



@patch
def clearUnknowns(self:CountryDetails):
    clearC = StyleCSS("blank",fill="none",stroke="none")
    self.countryMap.hexGrid.builder.add_style(clearC)
    for i in range(len(self.countryMap.hexGrid.hexes)):
        if self.regionMapper(i) < 0:
            self.countryMap.hexGrid.hexes[i].style = clearC

## Board

In [ ]:
#| export
class GameBoard:
    """ This lets us keep track of many of the games global properties"""

    def __init__(self,terrain,top_n=3,year=1900,gender=None):
        self.terrain = terrain
        self.world = Geology(terrain,plates=[])
        self.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in self.world.basins.sheds]
        self.shedScores.sort(key=lambda x: x[1], reverse=True)

        watershed_map = np.full(len(terrain.elevations), -1)  # -1 = no watershed
        for score in enumerate(self.shedScores):
            ws_id, watFlow = score
            watershed = watFlow[0]
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id

        self.watershed_map = watershed_map

        # mark the map where we can have things
        terrain = self.terrain
        countries = np.zeros(len(terrain.elevations))
        for i in range(len(terrain.elevations)):
            if terrain.elevations[i] < 1:
                countries[i] = -1
            if terrain.elevations[i] > terrain.elevationDelta * (len(terrain.colorLevels)-2):
                countries[i] = -1
        
        if len(self.world.basins.sheds) < 0:
            if debug:
                print("no water")
            self.kingdoms = []
            return

        kingdoms = Kingdom.start(self,self.shedScores[:top_n])
        for i, country in enumerate(kingdoms):

            country.countryName =  GameBoard.completeCountry(country.flag.name,country.flag.countryPrefix)
            for h in country.region:
                countries[h] = country.countryId

        terrain.fields["country"] = countries

        self.kingdoms = kingdoms

    @classmethod
    def completeCountry(cls, name ,place, pattern=None, descriptor=None):
        

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)
    
    @staticmethod
    def decode(s: str) -> 'GameBoard':
        """Decode GameBoard from string format."""
        lines = s.strip().split('\n')
        
        world = None
        kingdoms = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('+world'):
                world_lines = []
                i += 1
                while not lines[i].startswith('-world'):
                    world_lines.append(lines[i])
                    i += 1
                world = Geology.decode('\n'.join(world_lines))
            
            elif line.startswith('+kingdoms:'):
                i += 1
                while not lines[i].startswith('-kingdoms'):
                    if lines[i].startswith('+kingdom'):
                        kingdom_lines = []
                        i += 1
                        while not lines[i].startswith('-kingdom'):
                            kingdom_lines.append(lines[i])
                            i += 1
                        kingdoms.append(Kingdom.decode('\n'.join(kingdom_lines), world))
                    i += 1
            
            i += 1
        
        # Create GameBoard without triggering __init__
        board = object.__new__(GameBoard)
        board.terrain = world.terrain
        board.world = world
        board.kingdoms = kingdoms
        
        # Reconstruct derived fields
        board.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1]) for ws in world.basins.sheds]
        board.shedScores.sort(key=lambda x: x[1], reverse=True)
        
        # Rebuild watershed_map
        watershed_map = np.full(len(board.terrain.elevations), -1)
        for ws_id, watFlow in enumerate(board.shedScores):
            watershed = watFlow.watershed
            for hex_idx in watershed.region.hexes:
                watershed_map[hex_idx] = ws_id
        board.watershed_map = watershed_map
        
        return board

In [ ]:
#| export
@patch
def encode(self: GameBoard) -> str:
    """Encode GameBoard to string format."""
    lines = []
    
    # We encode world (which contains terrain and basins)
    lines.append("+world")
    lines.append(self.world.encode())
    lines.append("-world")
    
    # Kingdoms
    lines.append(f"+kingdoms:{len(self.kingdoms)}")
    for kingdom in self.kingdoms:
        lines.append("+kingdom")
        lines.append(kingdom.encode())
        lines.append("-kingdom")
    lines.append("-kingdoms")
    
    return '\n'.join(lines)

#| export
I would like to encode and decode GameBoard

In [ ]:
def demoKingdoms():

    sampleMap = TerraDemo().aussie_map()
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleWorld = GameBoard(sampleMap,top_n=7)
    
    return sampleWorld

ourBoard = demoKingdoms()
#print(countries)
for country in ourBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

aussieText = ourBoard.encode()

In [ ]:
demoBoard = GameBoard.decode(aussieText)
for country in demoBoard.kingdoms:
    
    print(f"{country.countryName} ruled by {country.flag.name} has {len(country.settlements)}")

In [ ]:
#| export
@patch
def explore(self: GameBoard, countryId) -> Watershed | None:
    country = [x for x in self.kingdoms if x.countryId == countryId][0]
    capital = country.settlements[0]
    grid = self.world.terrain.hexGrid
    countries = self.terrain.fields["country"]
    
    candidate_watersheds = []
    
    for direction in HexPosition.directions():
        spot = grid.index_to_hexposition(capital.location)
        place = capital.location
        
        # Walk in this direction until we leave the kingdom
        while 0 <= place < len(countries) and countries[place] == countryId:
            spot = spot + direction
            place = grid.hexposition_to_index(spot)
        
        # If we hit unclaimed land, check its watershed
        if 0 <= place < len(countries) and countries[place] == 0:
            ws_id = self.watershed_map[place]
            if ws_id >= 0:
                score = self.shedScores[ws_id]
                candidate_watersheds.append(score)
    
    # Return best watershed by flow, or None
    if candidate_watersheds:
        return max(candidate_watersheds, key=lambda x: x[1])
    return None


In [ ]:
#| export
@patch
def expand_kingdoms(self: GameBoard, max_rounds: int = 100):
    """Expand all kingdoms round-robin style until no more growth possible."""
    countries = self.terrain.fields["country"]
    
    for round_num in range(max_rounds):
        any_growth = False
        
        for kingdom in self.kingdoms:
            score = self.explore(kingdom.countryId)
            
            if score is not None:
                any_growth = True
                # Claim all hexes in this watershed
                for hex_idx in score.watershed.region.hexes:
                    countries[hex_idx] = kingdom.countryId
                    kingdom.region.hexes.add(hex_idx)
                kingdom.captured.append(score)
        
        # Stop if no kingdom could grow
        if not any_growth:
            print(f"Expansion complete after {round_num + 1} rounds")
            break
    
    return self.kingdoms


### Drawing

In [ ]:
#| export
@patch
def find_adjacent_kingdoms(self: Kingdom, all_kingdoms: list['Kingdom']) -> list[int]:
    """Find kingdoms that share a border with this one."""
    adjacent = []
    outside_hexes = self.region.outside(ring=1)
    
    for i, other in enumerate(all_kingdoms):
        if other is self:
            continue
        
        # Check if any of our outside hexes are in their region
        if any(hex_idx in other.region.hexes for hex_idx in outside_hexes.hexes):
            adjacent.append(i)
    
    return adjacent




In [ ]:
#| export
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost




In [ ]:
#| export
@patch
def find_path_dijkstra(self: GameBoard, start_hex: int, end_hex: int) -> list[HexPosition] | None:
    """Find shortest path using Dijkstra's algorithm with elevation costs."""
    #import heapq this is installed in the header

    class BHeapItem:
        def __init__(self, cost, hex_pos):
            self.cost = cost
            self.hex_pos = hex_pos

        def __lt__(self, other):
            return self.cost < other.cost

    grid = self.terrain.hexGrid
    
    # Convert start to HexPosition (using start as origin)
    start_pos = grid.index_to_hexposition(start_hex, start_hex)  # (0,0,0)
    end_pos = grid.index_to_hexposition(end_hex, start_hex)
    
    # Priority queue: (cost, HexPosition)
    queue = [BHeapItem(0, start_pos)]
    costs = {start_pos: 0}
    came_from = {}
    
    while queue:
        item = heapq.heappop(queue)
        current_cost, current = item.cost, item.hex_pos
        
        if current == end_pos:
            # Reconstruct path
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start_pos)
            return list(reversed(path))
        
        # Skip if we've found a better path already
        if current_cost > costs.get(current, float('inf')):
            continue
        
        # Check all 6 neighbors
        for direction in HexPosition.directions():
            neighbor = current + direction
            
            cost = self.movement_cost(current, neighbor, start_hex)
            if cost == float('inf'):
                continue
            
            new_cost = current_cost + cost
            
            if new_cost < costs.get(neighbor, float('inf')):
                costs[neighbor] = new_cost
                came_from[neighbor] = current
                heapq.heappush(queue, BHeapItem(new_cost, neighbor))
    
    return None  # No path found

Any thoughts on where to go next?


In [ ]:
def califorina_place(top_n=5):

    sampleMap = TerraDemo().california_map().downsample_climate(0.25)
    sampleMap.carve_to_ocean(num_lakes=1)
    sampleMap.hexGrid.adjustRadius(10)
    sampleWorld = GameBoard(sampleMap,top_n=5)
    sampleWorld.expand_kingdoms(max_rounds=50)
    
    return sampleWorld

cali = califorina_place()

## Helpers

In [ ]:
#| export
@patch
def invert_mapper(result: ZoomResult) -> dict:
    """Build coarse_idx → [fine_indices] from a ZoomResult.
    Pass the result dict to overlay functions as c2f."""
    c2f = {}
    invalid = result.terrain.hexGrid.invalidRegion
    for fine_idx in range(len(result.terrain.elevations)):
        if fine_idx in invalid:
            continue
        coarse_idx = result.mapper(fine_idx)
        if coarse_idx >= 0:
            c2f.setdefault(coarse_idx, []).append(fine_idx)
    return c2f


In [ ]:
#| export
@patch
def project_region(region: HexRegion, target_grid: HexGrid, c2f: dict = None) -> HexRegion:
    """Project a coarse region onto a target grid via c2f mapping.
    If c2f is None (identity), returns the region unchanged."""
    if c2f is None:
        return region
    fine_hexes = set()
    for coarse_idx in region.hexes:
        if coarse_idx in c2f:
            fine_hexes.update(c2f[coarse_idx])
    return HexRegion(hexes=fine_hexes, hexGrid=target_grid)


def _map_point(coarse_idx: int, c2f: dict = None) -> int:
    """Map a single coarse index to a fine index (picks middle representative).
    If c2f is None (identity), returns coarse_idx unchanged."""
    if c2f is None:
        return coarse_idx
    fs = c2f.get(coarse_idx)
    if not fs:
        return -1
    return fs[len(fs) // 2]

## Tables

In [ ]:
#| export
@dataclass
class PieceRecord:
    id: str = ""
    kingdom_id: int = 0
    settlement_id: str = ""  # empty if not in a settlement
    location: int = 0
    owner_id: int = 0
    size: int = 100
    health: int = 100
    max_health: int = 100
    sight: int = 3
    memory: float = 0.8
    movement_range: int = 4
    harvest_goal: str = ""  # store enum value as string
    settle_progress: int = 0
    settle_threshold: int = 3
    world_id: int = 0


In [ ]:
#| export
@dataclass
class SettlementRecord:
    id: str = ""
    kingdom_id: int = 0
    location: int = 0
    name: str = ""
    size: int = 100
    health: int = 100
    world_id: int = 0


In [ ]:
#| export
@dataclass
class KingdomRecord:
    id: int = None          # auto-increment pk
    world_id: int = 0
    country_id: int = 0     # countryId within this world
    country_name: str = ""
    flag_data: str = ""     # CountryFlag.encode()
    capital_settlement_id: str = ""

@dataclass
class KingdomHex:
    id: int = None
    world_id: int = 0
    kingdom_id: int = 0     # matches country_id
    grid_index: int = 0
    q: int = 0
    r: int = 0
    s: int = 0


## Game Storage

In [ ]:
#| export
class GameStorage(GeoStorage):
    def createDB(self):
        super().createDB()

        # Pieces & Settlements
        self.db.create(PieceRecord, pk='id', if_not_exists=True, transform=True)
        self.pieces = self.db.t.piece_record
        self.db.create(SettlementRecord, pk='id', if_not_exists=True, transform=True)
        self.settlements = self.db.t.settlement_record

        # Kingdoms
        self.db.create(KingdomRecord, pk='id', if_not_exists=True, transform=True)
        self.kingdom_records = self.db.t.kingdom_record
        self.db.create(KingdomHex, pk='id', if_not_exists=True, transform=True)
        self.kingdom_hexes = self.db.t.kingdom_hex

        # Kingdom indices
        self.db.execute(
            "CREATE INDEX IF NOT EXISTS idx_kr_world "
            "ON kingdom_record(world_id)")
        self.db.execute(
            "CREATE INDEX IF NOT EXISTS idx_kr_country "
            "ON kingdom_record(world_id, country_id)")
        self.db.execute(
            "CREATE INDEX IF NOT EXISTS idx_kh_kingdom "
            "ON kingdom_hex(world_id, kingdom_id)")
        self.db.execute(
            "CREATE INDEX IF NOT EXISTS idx_kh_grid "
            "ON kingdom_hex(world_id, grid_index)")
        self.db.execute(
            "CREATE INDEX IF NOT EXISTS idx_kh_coords "
            "ON kingdom_hex(world_id, q, r, s)")


In [ ]:
#| export
@patch
def save(self: Piece, db: GameStorage = None, world_id: int = 0, 
         settlement_id: str = "", kingdom_id: int = 0):
    """Save this piece to the database as a PieceRecord."""
    db = db or self.db
    if db is None:
        raise ValueError("No database — pass db or set piece.db")
    
    record = PieceRecord(
        id=self.id,
        kingdom_id=kingdom_id or self.owner_id,
        settlement_id=settlement_id,
        location=self.location or -1,
        owner_id=self.owner_id,
        size=self.size,
        health=self.health,
        max_health=self.max_health,
        sight=self.sight,
        memory=self.memory,
        movement_range=self.movement_range,
        harvest_goal=self.harvest_goal.value if self.harvest_goal else "",
        settle_progress=self.settle_progress,
        settle_threshold=self.settle_threshold,
        world_id=world_id,
    )
    db.pieces.upsert(asdict(record), pk='id')
    return record


In [ ]:
#| export
@patch
def save(self: Settlement, world_id: int = 0):
    """Save this settlement and all its citizens to the database."""
    if self.db is None:
        raise ValueError("No database — pass db")
    db = self.db
    
    record = SettlementRecord(
        id=self.id,
        kingdom_id=self.owner_id,
        location=self.location or -1,
        name=self.name,
        size=self.size,
        health=self.health,
        world_id=world_id,
    )
    db.settlements.upsert(asdict(record), pk='id')
    
    for citizen in self.citizens:
        citizen.save(db=db, world_id=world_id,
                     settlement_id=self.id, kingdom_id=self.owner_id)
    return record


In [ ]:
#| export

@patch
def save(self: Kingdom, db: GameStorage = None, world_id: int = 0):
    """Save kingdom record, region hexes, and cascade to settlements."""
    db = db or self.db
    if db is None:
        raise ValueError("No database — pass db or set kingdom.db")

    grid = self.world.terrain.hexGrid
    capital = self.settlements[0] if self.settlements else None

    with db.db.conn:
        # 1. Upsert KingdomRecord — find existing by (world_id, country_id)
        existing = db.db.execute(
            "SELECT id FROM kingdom_record WHERE world_id = ? AND country_id = ?",
            [world_id, self.countryId]
        ).fetchone()

        record = KingdomRecord(
            id=existing[0] if existing else None,
            world_id=world_id,
            country_id=self.countryId,
            country_name=self.countryName,
            flag_data=self.flag.encode() if self.flag else "",
            capital_settlement_id=capital.id if capital else "",
        )
        #db.kingdom_records.upsert(asdict(record), pk='id')
        #print(asdict(record))
        d = asdict(record)
        if d['id'] is None:
            del d['id']
            db.kingdom_records.insert(d)
        else:
            db.kingdom_records.upsert(d, pk='id')


        # 2. Replace region hexes atomically
        db.db.execute(
            "DELETE FROM kingdom_hex WHERE world_id = ? AND kingdom_id = ?",
            [world_id, self.countryId]
        )
        for hex_idx in self.region.hexes:
            pos = grid.index_to_hexposition(hex_idx)
            db.kingdom_hexes.insert(asdict(KingdomHex(
                world_id=world_id,
                kingdom_id=self.countryId,
                grid_index=hex_idx,
                q=pos.q, r=pos.r, s=pos.s,
            )))

        # 3. Cascade to settlements (which cascade to pieces)
        for s in self.settlements:
            s.db = db
            s.owner_id = self.countryId
            s.save(world_id=world_id)

    return record


In [ ]:
#| export
@patch
def save(self: GameBoard, db: GameStorage, world_id: int):
    """Save all kingdoms (and their settlements/pieces) to the database."""
    for kingdom in self.kingdoms:
        kingdom.db = db
        kingdom.save(db=db, world_id=world_id)

### Fetching

In [ ]:
#| export
@patch
def piece_from_id(db: GameStorage, piece_id: str) -> 'Piece':
    """Load a Piece from the database by ID."""
    row = db.pieces[piece_id]
    if row is None:
        raise KeyError(f"Piece {piece_id} not found")

    r = row if isinstance(row, dict) else dict(row)
    return Piece(
        id=r['id'],
        owner_id=r['owner_id'],
        location=r['location'] if r['location'] >= 0 else None,
        size=r['size'],
        health=r['health'],
        max_health=r['max_health'],
        sight=r['sight'],
        memory=r['memory'],
        movement_range=r['movement_range'],
        harvest_goal=Resources(r['harvest_goal']) if r['harvest_goal'] else None,
        settle_progress=r['settle_progress'],
        settle_threshold=r['settle_threshold'],
    )



In [ ]:
#| export
@patch
def settlement_from_id(db: GameStorage, settlement_id: str) -> 'Settlement':
    """Load a Settlement and its citizens from the database."""
    row = db.settlements[settlement_id]
    if row is None:
        raise KeyError(f"Settlement {settlement_id} not found")

    r = row if isinstance(row, dict) else dict(row)

    # Load citizens from piece_record
    cursor = db.db.execute(
        "SELECT id FROM piece_record WHERE settlement_id = ?", [settlement_id]
    )
    citizen_ids = [row[0] for row in cursor.fetchall()]
    citizens = [db.piece_from_id( pid) for pid in citizen_ids]

    s = Settlement(
        id=r['id'],
        owner_id=r['kingdom_id'],
        location=r['location'] if r['location'] >= 0 else None,
        name=r['name'],
        size=r['size'],
        health=r['health'],
        citizens=citizens,
    )
    s.db = db
    return s




In [ ]:
#| export
@patch
def kingdom_from_db(store:GameStorage , country_id: int, world_id: int, world) -> 'Kingdom':
    """Load a Kingdom from the database.
    
    Args:
        db: GameStorage instance
        country_id: the countryId (not the auto-increment pk)
        world_id: which world
        world: object with .terrain (Geology or GameBoard)
    """
    # Find the record
    row = store.db.execute(
        "SELECT * FROM kingdom_record WHERE world_id = ? AND country_id = ?",
        [world_id, country_id]
    ).fetchone()
    if row is None:
        raise KeyError(f"Kingdom country_id={country_id} not found in world {world_id}")

    cols = [c.name for c in store.kingdom_records.columns]
    r = dict(zip(cols, row))

    grid = world.terrain.hexGrid

    # Load region hexes
    cursor = store.db.execute(
        "SELECT grid_index FROM kingdom_hex WHERE world_id = ? AND kingdom_id = ?",
        [world_id, country_id]
    )
    hex_indices = set(row[0] for row in cursor.fetchall())
    region = HexRegion(hexes=hex_indices, hexGrid=grid)

    # Load settlements
    cursor = store.db.execute(
        "SELECT id FROM settlement_record WHERE world_id = ? AND kingdom_id = ?",
        [world_id, country_id]
    )
    settlement_ids = [row[0] for row in cursor.fetchall()]
    settlements = [store.settlement_from_id( sid) for sid in settlement_ids]

    # Decode flag
    flag = CountryFlag.decode(r['flag_data']) if r['flag_data'] else None
    if flag is not None:
        for settle in settlements:
            settle.updateFlag(flag)

    # Build kingdom — use first settlement location as capital_hex
    capital_hex = settlements[0].location if settlements else 0
    kingdom = Kingdom(capital_hex, region, world, countryId=country_id, flag=flag)
    kingdom.settlements = settlements
    kingdom.countryName = r['country_name']
    kingdom.db = store

    return kingdom



In [ ]:
#| export
@patch
def gameboard(store: GameStorage, world_id: int, terrain: Terrain) -> 'GameBoard':
    """Load a GameBoard from the database, given terrain already loaded."""
    board = object.__new__(GameBoard)
    board.terrain = terrain
    board.world = Geology(terrain, plates=[])

    # Recompute watershed scores (needed for explore/expand)
    board.shedScores = [WatershedFlow(ws, ws.max_flow_hex()[1])
                        for ws in board.world.basins.sheds]
    board.shedScores.sort(key=lambda x: x[1], reverse=True)

    # Rebuild watershed_map
    watershed_map = np.full(len(terrain.elevations), -1)
    for ws_id, watFlow in enumerate(board.shedScores):
        for hex_idx in watFlow.watershed.region.hexes:
            watershed_map[hex_idx] = ws_id
    board.watershed_map = watershed_map

    # Load all kingdoms for this world
    cursor = store.db.execute(
        "SELECT country_id FROM kingdom_record WHERE world_id = ?", [world_id]
    )
    country_ids = [row[0] for row in cursor.fetchall()]
    board.kingdoms = [store.kingdom_from_db(cid, world_id, board) for cid in country_ids]

    # Rebuild country field from DB
    store.load_country_field(terrain, world_id)

    return board



In [ ]:
@patch
def kingdom_detail(self: GameStorage, world_id: int, country_id: int, 
                   terrain: Terrain, scale: int = 2, 
                   rings: int = 5, halo_rings: int = 1) -> ZoomResult:
    """Zoom into a kingdom's region for high-resolution terrain."""
    grid = terrain.hexGrid
    
    # 1. Load the kingdom's region
    cursor = self.db.execute(
        "SELECT grid_index FROM kingdom_hex WHERE world_id = ? AND kingdom_id = ?",
        [world_id, country_id]
    )
    hex_indices = set(row[0] for row in cursor.fetchall())
    region = HexRegion(hexes=hex_indices, hexGrid=grid)
    
    # 2. Build a ChunkCover and attach storage
    cover = ChunkCover(terrain, rings=rings, halo_rings=halo_rings)
    cover.db = self
    cover.save(name=f"kingdom_{country_id}")
    
    # 3. Zoom into the kingdom's region
    return cover.zoom_region(region, scale=scale, compute_weather=True)


#### Ownership

In [ ]:
#| export
@patch
def hex_owner(self: GameStorage, world_id: int, grid_index: int) -> int:
    """Return kingdom_id that owns this hex, or 0 if unclaimed."""
    row = self.db.execute(
        "SELECT kingdom_id FROM kingdom_hex WHERE world_id = ? AND grid_index = ?",
        [world_id, grid_index]
    ).fetchone()
    return row[0] if row else 0

@patch
def load_country_field(self: GameStorage, terrain: Terrain, world_id: int):
    """Rebuild terrain.fields['country'] from kingdom_hex table."""
    countries = np.zeros(len(terrain.elevations))

    # Mark water/mountains as unavailable
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] < 1:
            countries[i] = -1
        elif terrain.elevations[i] > terrain.elevationDelta * (len(terrain.colorLevels) - 2):
            countries[i] = -1

    cursor = self.db.execute(
        "SELECT grid_index, kingdom_id FROM kingdom_hex WHERE world_id = ?",
        [world_id]
    )
    for row in cursor.fetchall():
        idx, kid = row[0], row[1]
        if 0 <= idx < len(countries):
            countries[idx] = kid

    terrain.fields["country"] = countries


### Overlays

In [ ]:
SVGBuilder.BUILDERHIDE = True

#| export
@patch
def pieceOverlay(self: GameStorage, terrain: Terrain, world_id: int, 
                 mapper: Callable[[int], int] = None) -> str:
    """Draw pieces from the database onto a terrain map."""
    if mapper is None:
        mapper = lambda idx: idx
    
    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    terrain.hexGrid.builder.add_style(pieceStyle)
    
    # Query all pieces for this world — grab cursor before fetchall
    cursor = self.db.execute(
        "SELECT * FROM piece_record WHERE world_id = ?", [world_id]
    )
    cols = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    
    if not rows:
        return ""
    
    pieces = [dict(zip(cols, row)) for row in rows]
    
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)
    retStr = ""
    
    for p in pieces:
        loc = p.get('location')
        if loc is None or loc < 0:
            continue
        
        local_idx = mapper(loc)
        if local_idx < 0 or local_idx >= num_hexes:
            continue
        
        coords = grid.hexes[local_idx].center
        retStr += (
            f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
            f"class='{pieceStyle.name}'></circle>\n"
        )
    
    return retStr


# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:

    dbg.server = GameStorage(custom_path=dbg.db_path)
  
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    #els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    #print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    #for i in range(len(terrain.elevations)):
    #    if i not in region:
    #        terrain.elevations[i] = 3000
    #terrain.textElevations()
    print("now detail")

    # result.basins has watersheds computed on the fine (zoomed) grid
    # Score them and pick the best — it corresponds to our target watershed
    fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
    fine_scores.sort(key=lambda x: x[1], reverse=True)

    # The top watershed in the zoomed region is our target
    best_ws = fine_scores[0][0]
    capital_hex = best_ws.max_flow_hex()[0]

    print(f"Capital hex: {capital_hex}")
    print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
    piece = Piece(location=capital_hex, owner_id=1, current_position=capital_hex)
    piece.db = dbg.server
    piece.save(world_id=cover.ident)
   



    
    # 6. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    #capital = ws.max_flow_hex()[0]

    #result.terrain.textElevations()
    terrain = result.terrain
    over = dbg.server.pieceOverlay(result.terrain, world_id=cover.ident)
    
    terrain.hexGrid.builder.adjust("pieces",over)
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()

So settements has to have a similar kind of design

#| export
@patch
def settlementOverlay(self: GameStorage, terrain: Terrain, world_id: int, 
                 mapper: Callable[[int], int] = None) -> str:
    """Draw settlements from the database onto a terrain map."""
    if mapper is None:
        mapper = lambda idx: idx
    
    settlementStyle = StyleCSS("settlementStyle", fill="purple", stroke="black")
    terrain.hexGrid.builder.add_style(settlementStyle)
    
    # Query settlements (not pieces!) — use fetchall() immediately
    rows = list(self.db.execute(
        "SELECT * FROM settlement_record WHERE world_id = ?", [world_id]
    ).fetchall())
    
    if not rows:
        return ""
    
    # Get column names from the table schema instead of cursor.description
    cols = [col.name for col in self.settlements.columns]
    settlements = [dict(zip(cols, row)) for row in rows]
    
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)
    retStr = ""
    
    for s in settlements:
        loc = s.get('location')
        if loc is None or loc < 0:
            continue
        
        local_idx = mapper(loc)
        if local_idx < 0 or local_idx >= num_hexes:
            continue
        
        coords = grid.hexes[local_idx].center
        retStr += (
            f"<circle cx='{coords.x}' cy='{coords.y}' r='30' "
            f"class='{settlementStyle.name}'></circle>\n"
        )
    
    return retStr


#| export
# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:

    dbg.server = GameStorage(custom_path=dbg.db_path)
  
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    #els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    #print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    #for i in range(len(terrain.elevations)):
    #    if i not in region:
    #        terrain.elevations[i] = 3000
    #terrain.textElevations()
    print("now detail")

    # result.basins has watersheds computed on the fine (zoomed) grid
    # Score them and pick the best — it corresponds to our target watershed
    fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
    fine_scores.sort(key=lambda x: x[1], reverse=True)

    # The top watershed in the zoomed region is our target
    best_ws = fine_scores[0][0]
    capital_hex = best_ws.max_flow_hex()[0]

    print(f"Capital hex: {capital_hex}")
    print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
    capital = Settlement(location=capital_hex, owner_id=1)
    capital.db = dbg.server
    capital.save(world_id=cover.ident)
   

    # 6. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    #capital = ws.max_flow_hex()[0]

    #result.terrain.textElevations()
    terrain = result.terrain
    over = dbg.server.settlementOverlay(result.terrain, world_id=cover.ident)
    
    terrain.hexGrid.builder.adjust("Settlement",over)
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()

#| export
@patch
def countries_overlay(self: GameBoard) -> str:
    """Create overlay showing kingdom territories with windy borders.
    """
   
    overlay = ""
    borders = {}  # Shared border cache
    terrain = self.terrain
    grid = terrain.hexGrid

    for  country in self.kingdoms:
        
        
        style = country.flag.kingStyle(f"{country.flag.name}_{country.countryId}")
        #print(StyleCSS.generate([style]))
        grid.builder.add_style(style)

        for path in country.region.trace_perimeter_cached(
            borders,
            style=style,
            f=unique_windy_edge(iterations=2, offset_min=0.05, offset_max=0.15)
        ):
            overlay += path.svg()

    return overlay


#| export
@patch
def names_overlay(self: GameBoard) -> str:
    """Create overlay showing kingdom territories with windy borders and labels."""
    terrain = self.terrain
    grid = terrain.hexGrid
    
    
    
    overlay = ""
    for country in self.kingdoms:
        # Add label style
        label_style =  country.flag.labelStyle(f"n_{country.countryId}")
        label_style.properties["stroke"] = "#36454F"
        label_style.properties["fill"] = "none"
        #print(StyleCSS.generate([label_style]))
        grid.builder.add_style(label_style)
        if len(country.region.hexes) > 0:
            # Add country name at centroid
            centroid_idx = country.region.centroid_hex()
            if centroid_idx >= 0 and country.countryName:
                hex_obj = grid.hexes[centroid_idx]
                cx, cy = hex_obj.center.x, hex_obj.center.y
                overlay += f'\t<text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="middle" class="{label_style.name}">{country.countryName}</text>\n'

    return overlay

### At scale

In [ ]:
#| export
@patch
def countries_overlay(self: GameBoard, terrain: Terrain = None, 
                      c2f: dict = None) -> str:
    """Kingdom borders. Pass terrain + c2f from invert_mapper for zoomed views."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    overlay = ""
    borders = {}

    for country in self.kingdoms:
        region = country.region.project_region( grid, c2f)
        if not region.hexes:
            continue

        style = country.flag.kingStyle(f"{country.flag.name}_{country.countryId}")
        grid.builder.add_style(style)

        for path in region.trace_perimeter_cached(
            borders, style=style,
            f=unique_windy_edge(iterations=2, offset_min=0.05, offset_max=0.15)
        ):
            overlay += path.svg()

    return overlay


In [ ]:
#| export
@patch
def names_overlay(self: GameBoard, terrain: Terrain = None,
                  c2f: dict = None) -> str:
    """Kingdom name labels."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    overlay = ""

    for country in self.kingdoms:
        if not country.countryName:
            continue

        region = country.region.project_region( grid, c2f)
        if not region.hexes:
            continue

        label_style = country.flag.labelStyle(f"n_{country.countryId}")
        label_style.properties["stroke"] = "#36454F"
        label_style.properties["fill"] = "none"
        grid.builder.add_style(label_style)

        centroid_idx = region.centroid_hex()
        if centroid_idx >= 0:
            cx = grid.hexes[centroid_idx].center.x
            cy = grid.hexes[centroid_idx].center.y
            overlay += (
                f'\t<text x="{cx}" y="{cy}" text-anchor="middle" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{country.countryName}</text>\n'
            )

    return overlay


In [ ]:
#| export
@patch
def settlementOverlay(self: GameBoard, terrain: Terrain = None,
                      c2f: dict = None) -> str:
    """Settlement markers."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    

    overlay = ""
    for i, country in enumerate(self.kingdoms):
        if country.flag is None:
            fill = "purple"
            print(f"{country.countryName} has no flag")
        else:
            fill = country.flag.tri1
            print(f"{country.countryName} has {country.flag.tri1} {country.flag.tri2} ")
        settleStyle = StyleCSS(f"settlementStyle_{i}", fill=fill, stroke="black")
        grid.builder.add_style(settleStyle)

        for s in country.settlements:
            if s.location is None:
                continue
            local_idx = _map_point(s.location, c2f)
            if local_idx < 0 or local_idx >= num_hexes:
                continue
            coords = grid.hexes[local_idx].center
            overlay += (
                f"<circle cx='{coords.x}' cy='{coords.y}' r='20' "
                f"class='{settleStyle.name}'></circle>\n"
            )

        print(settleStyle)

    return overlay


# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:

    dbg.server = GameStorage(custom_path=dbg.db_path)
  
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    #els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    #print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    #for i in range(len(terrain.elevations)):
    #    if i not in region:
    #        terrain.elevations[i] = 3000
    #terrain.textElevations()
    print("now detail")

    # result.basins has watersheds computed on the fine (zoomed) grid
    # Score them and pick the best — it corresponds to our target watershed
    fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
    fine_scores.sort(key=lambda x: x[1], reverse=True)

    # The top watershed in the zoomed region is our target
    best_ws = fine_scores[0][0]
    capital_hex = best_ws.max_flow_hex()[0]

    print(f"Capital hex: {capital_hex}")
    print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
    capital = Settlement(location=capital_hex, owner_id=1)
    capital.db = dbg.server
    capital.save(world_id=cover.ident)
   

    # 6. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    #capital = ws.max_flow_hex()[0]

    #result.terrain.textElevations()
    terrain = result.terrain
    over = dbg.server.settlementOverlay(result.terrain, world_id=cover.ident)
    
    terrain.hexGrid.builder.adjust("Settlement",over)
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()

In [ ]:
@patch
def pieceOverlay(self: GameBoard, terrain: Terrain = None,
                 c2f: dict = None) -> str:
    """Piece markers. Pass terrain + c2f from invert_mapper for zoomed views."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    grid.builder.add_style(pieceStyle)

    overlay = ""
    for country in self.kingdoms:
        for s in country.settlements:
            for piece in s.citizens:
                if piece.location is None:
                    continue
                local_idx = _map_point(piece.location, c2f)
                if local_idx < 0 or local_idx >= num_hexes:
                    continue
                coords = grid.hexes[local_idx].center
                overlay += (
                    f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
                    f"class='{pieceStyle.name}'></circle>\n"
                )

    return overlay


# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:

    dbg.server = GameStorage(custom_path=dbg.db_path)
  
    # 1. Load Sydney terrain
    terrain = TerraDemo().bayArea_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    #els =  [terrain.elevations[i] for i in target_shed.region.hexes]
    #print(els)
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    #for i in range(len(terrain.elevations)):
    #    if i not in region:
    #        terrain.elevations[i] = 3000
    #terrain.textElevations()
    print("now detail")

    # result.basins has watersheds computed on the fine (zoomed) grid
    # Score them and pick the best — it corresponds to our target watershed
    fine_scores = [(ws, ws.max_flow_hex()[1]) for ws in result.basins.sheds]
    fine_scores.sort(key=lambda x: x[1], reverse=True)

    # The top watershed in the zoomed region is our target
    best_ws = fine_scores[0][0]
    capital_hex = best_ws.max_flow_hex()[0]

    print(f"Capital hex: {capital_hex}")
    print(f"Watershed has {len(best_ws.region.hexes)} hexes in zoomed grid")
    piece = Piece(location=capital_hex, owner_id=1, current_position=capital_hex)
    piece.db = dbg.server
    piece.save(world_id=cover.ident)
   
    
    # 6. Visualize
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    #capital = ws.max_flow_hex()[0]

    #result.terrain.textElevations()
    terrain = result.terrain
    over = dbg.server.pieceOverlay(result.terrain, world_id=cover.ident)
    
    terrain.hexGrid.builder.adjust("pieces",over)
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()

## Building

In [ ]:
SVGBuilder.BUILDERHIDE = False

In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:
SVGBuilder.BUILDERHIDE = False

Do we have enought to show a gameboard from the bayarea?

In [ ]:
# Bay Area kingdoms
terrain = TerraDemo().bayArea_map()
terrain.carve_to_ocean(num_lakes=1)
terrain.hexGrid.adjustRadius(10)

board = GameBoard(terrain, top_n=5)
board.expand_kingdoms(max_rounds=50)

# Paint + overlay
terrain.colorMap()
terrain.hexGrid.update()

borders = board.countries_overlay()
names = board.names_overlay()

terrain.hexGrid.builder.adjust("kingdoms", borders)
terrain.hexGrid.builder.adjust("names", names)

for k in board.kingdoms:
    print(f"  {k.countryName}: {len(k.region.hexes)} hexes, {len(k.settlements)} settlements")

terrain.hexGrid.builder.show()


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    dbg.server = GameStorage(custom_path=dbg.db_path)
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area Kingdoms")
    
    print(f"world_id = {cover.ident}")
    board.save(dbg.server, world_id=cover.ident)
    
    # Debug: what's actually in the tables?
    kr = dbg.server.db.execute("SELECT COUNT(*) FROM kingdom_record").fetchone()[0]
    kh = dbg.server.db.execute("SELECT COUNT(*) FROM kingdom_hex").fetchone()[0]
    sr = dbg.server.db.execute("SELECT COUNT(*) FROM settlement_record").fetchone()[0]
    print(f"kingdom_records: {kr}, kingdom_hexes: {kh}, settlements: {sr}")
    
    # Check world_id match
    rows = dbg.server.db.execute("SELECT id, world_id, country_id, country_name FROM kingdom_record").fetchall()
    for r in rows:
        print(f"  KR: id={r[0]} world_id={r[1]} country_id={r[2]} name={r[3]}")
    
    loaded = dbg.server.gameboard(cover.ident, terrain)
    print(f"\nLoaded {len(loaded.kingdoms)} kingdoms from DB")


can we do something like CountryDetails. I am going to deprecate the class, but I do want to be able to generate the detailed terrain of a kingdom easily using our new storage

I would love to get a higher resolution of the kindom map like ChunkCover.zoom_region

lets hang it off of store right now. My thoughts are we have store for terrain details and GameBoard for mechanics of the game

In [ ]:
!cat ../../HexMagic/database.py

In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    dbg.server = GameStorage(custom_path=dbg.db_path)
    
    terrain = TerraDemo().bayArea_map()
    terrain.carve_to_ocean(num_lakes=1)
    terrain.hexGrid.adjustRadius(10)
    
    board = GameBoard(terrain, top_n=5)
    board.expand_kingdoms(max_rounds=50)
    
    # Save everything
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Bay Area")
    board.save(dbg.server, world_id=cover.ident)
    
    # Zoom into first kingdom
    k = board.kingdoms[0]
    result = dbg.server.kingdom_detail(cover.ident, k.countryId, terrain)
    
    result.terrain.hexGrid.adjustRadius(terrain.hexGrid.radius)
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    
    c2f = result.invert_mapper()


    borders = board.countries_overlay(result.terrain, c2f)
    names   = board.names_overlay(result.terrain, c2f)
    settle  = board.settlementOverlay(result.terrain, c2f)
    builder = result.terrain.hexGrid.builder
    builder.adjust("borders",borders)
    builder.adjust("names",names)
    builder.adjust("settle",settle)
    

    
    
    print(f"{k.countryName}: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
result.terrain.hexGrid.builder.show()


So I 

I catted the file. Does ZoomResult make sense?

can you write it?

Should we refactor : names_overlay, countries_overlay & settlement_overlay so they take in a masterTerrain and a mapper. If we wanted to do the whole map we could have it default to the identiy function

what went wrong

Pieces overlay should also take  c2f: dict = None)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — it should match the same signature as `settlementOverlay`. You'd want something like:

```python
@patch
def pieceOverlay(self: GameBoard, terrain: Terrain = None,
                 c2f: dict = None) -> str:
```

Then inside, use `_map_point(piece.location, c2f)` the same way `settlementOverlay` does. The rest of the logic (iterating `self.kingdoms` → each kingdom's settlements → each settlement's citizens, drawing circles) stays the same.

Want me to write it out, or do you want to take a crack at it?

can you write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
@patch
def pieceOverlay(self: GameBoard, terrain: Terrain = None,
                 c2f: dict = None) -> str:
    """Piece markers. Pass terrain + c2f from invert_mapper for zoomed views."""
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    pieceStyle = StyleCSS("piece", fill="red", stroke="black")
    grid.builder.add_style(pieceStyle)

    overlay = ""
    for country in self.kingdoms:
        for s in country.settlements:
            for piece in s.citizens:
                if piece.location is None:
                    continue
                local_idx = _map_point(piece.location, c2f)
                if local_idx < 0 or local_idx >= num_hexes:
                    continue
                coords = grid.hexes[local_idx].center
                overlay += (
                    f"<circle cx='{coords.x}' cy='{coords.y}' r='10' "
                    f"class='{pieceStyle.name}'></circle>\n"
                )

    return overlay
```

Same pattern as `settlementOverlay` — defaults to the identity when `c2f` is `None`, otherwise projects through `_map_point`. Note this only finds pieces that are inside settlements' `citizens` lists. If you also have free-roaming pieces (not attached to any settlement), you'd need a separate list on `Kingdom` or `GameBoard` to iterate over those too.